## Evaluation of embeddings

All evaluation methods are intrinsic, as the embeddings were trained for academic purposes. The evaluation
 includes word similarity benchmarks such as WordSim353 and word analogy tests covering syntactic and semantic relations. Cosine distance is used as a distance metric.

In [1]:
import bokeh.models as bm
import bokeh.plotting as bplt
import numpy as np
import polars as pl
from bokeh.io import output_notebook, output_file, save
from sklearn.manifold import TSNE
from scipy.stats import spearmanr

from embedding import Word2Vec, cosine_dist

output_notebook()

Loading BokehJS ...

In [5]:
SYNTACTIC_ANALOGIES = [row[2:] for row in pl.read_csv('data/msr.csv').iter_rows()]
SEMANTIC_ANALOGIES = [row[2:] for row in pl.read_csv('data/semeval.csv').iter_rows()]
WORDSIM_353_REL = [row[1:] for row in pl.read_csv('data/wordsim353-rel.csv').iter_rows()]
WORDSIM_353_SIM = [row[1:] for row in pl.read_csv('data/wordsim353-sim.csv').iter_rows()]

In [6]:
def draw_semantic_space(
        x: np.ndarray,
        y: np.ndarray,
        title="",
        radius=10,
        alpha=0.25,
        width=700,
        height=700,
        show: bool = True,
        **kwargs
):
    data_source = bm.ColumnDataSource({
        "x": x.tolist(),
        "y": y.tolist(),
        **kwargs
    })
    fig = bplt.figure(active_scroll='wheel_zoom', width=width, height=height, title=title)
    fig.scatter(
        "x", "y",
        size=radius,
        alpha=alpha,
        source=data_source
    )
    fig.add_tools(
        bm.HoverTool(tooltips=[(key, '@' + key) for key in kwargs.keys()])
    )

    if show:
        bplt.show(fig)
    return fig

def draw_embeddings(word2vec: Word2Vec, perplexity: int, html_file_name: str = None):
    points = TSNE(perplexity=perplexity, metric='cosine').fit_transform(word2vec.embeddings)
    if html_file_name is not None:
        output_file(filename=f'data/{html_file_name}.html')
    figure = draw_semantic_space(points[:, 0], points[:, 1], label=word2vec.words)
    if html_file_name is not None:
        save(figure)

def mean_dist_from_average_vector(word2vec: Word2Vec):
    mean_vector = word2vec.embeddings.mean(axis=0)
    return cosine_dist(mean_vector, word2vec.embeddings).mean()

def mean_pairwise_dist(word2vec: Word2Vec) -> np.ndarray:
    dot_products = word2vec.embeddings @ word2vec.embeddings.T
    norms = np.linalg.norm(word2vec.embeddings, axis=1, keepdims=True)
    norms = norms @ norms.T
    cos_dist = 1.0 - dot_products / norms
    upper_triangle = np.triu_indices(word2vec.vocab_size, 1)
    return cos_dist[upper_triangle].mean()

def eval_word_analogies(word2vec: Word2Vec, analogy_pairs: list[tuple[str, str, str, str]]):
    results = []
    for word1, word2, word3, target in analogy_pairs:
        known_words = (word1 in word2vec) and (word2 in word2vec) and (word3 in word2vec) and (target in word2vec)
        if not known_words:
            continue
        vector = word2vec[word2] - word2vec[word1] + word2vec[word3]
        found = word2vec.get_neighbors(vector=vector, k=1)[0][0]
        results.append(found == target)
    return np.array(results, dtype=np.bool).mean()

def eval_word_similarities(word2vec: Word2Vec, word_pairs: list[tuple[str, str, float]]) -> tuple[float, float]:
    expected = []
    actual = []
    for word1, word2, score in word_pairs:
        known_words = (word1 in word2vec) and (word2 in word2vec)
        if not known_words:
            continue
        expected.append(score)
        actual.append(word2vec[word1] @ word2vec[word2].T)
    return spearmanr(expected, actual).statistic, len(expected) / len(word_pairs)

def print_stat(word2vec: Word2Vec):
    print(f'Distance from mean vector: {mean_dist_from_average_vector(word2vec).mean():.3f}')
    print(f'Pairwise distance: {mean_pairwise_dist(word2vec):.3f}')
    print('----')
    print(f'Syntactic analogies score {eval_word_analogies(word2vec, SYNTACTIC_ANALOGIES).mean():.3f}')
    print(f'Semantic analogies score: {eval_word_analogies(word2vec, SEMANTIC_ANALOGIES).mean():.3f}')
    print('----')
    wordsim_rel_score, wordsim_rel_cover = eval_word_similarities(word2vec, WORDSIM_353_REL)
    print(f'wordsim-rel spearman corr: {wordsim_rel_score:.3f}, coverage: {wordsim_rel_cover:.3f}')
    wordsim_sim_score, wordsim_sim_cover = eval_word_similarities(word2vec, WORDSIM_353_SIM)
    print(f'wordsim-sim spearman corr: {wordsim_sim_score:.3f}, coverage: {wordsim_sim_cover:.3f}')


#### Baseline
Random vectors sampled from a uniform distribution

In [7]:
baseline = Word2Vec.from_pretrained('data/default_window')
baseline._embeddings = np.random.uniform(-5e-3, 5e-3, size=(baseline.vocab_size, baseline.embedding_dim))
print_stat(baseline)

Distance from mean vector: 0.992
Pairwise distance: 1.000
----
Syntactic analogies score 0.000
Semantic analogies score: 0.002
----
wordsim-rel spearman corr: -0.040, coverage: 0.810
wordsim-sim spearman corr: 0.017, coverage: 0.760


#### SGNS trained with a context window size of 5

In [9]:
window5 = Word2Vec.from_pretrained('data/default_window')
print_stat(window5)

Distance from mean vector: 0.467
Pairwise distance: 0.716
----
Syntactic analogies score 0.023
Semantic analogies score: 0.002
----
wordsim-rel spearman corr: 0.336, coverage: 0.810
wordsim-sim spearman corr: 0.552, coverage: 0.760


In [19]:
draw_embeddings(window5, 30, html_file_name='word2vec_w3_p30')

#### SGNS trained a context window size of 5

In [11]:
window3 = Word2Vec.from_pretrained('data/small_window')
print_stat(window3)

Distance from mean vector: 0.209
Pairwise distance: 0.374
----
Syntactic analogies score 0.031
Semantic analogies score: 0.004
----
wordsim-rel spearman corr: 0.028, coverage: 0.810
wordsim-sim spearman corr: 0.290, coverage: 0.760


In [18]:
draw_embeddings(window3, 40, html_file_name='word2vec_w5_p40')

#### SGNS trained a context window size of 7

In [13]:
window7 = Word2Vec.from_pretrained('data/big_window')
print_stat(window7)

Distance from mean vector: 0.355
Pairwise distance: 0.584
----
Syntactic analogies score 0.029
Semantic analogies score: 0.002
----
wordsim-rel spearman corr: 0.292, coverage: 0.810
wordsim-sim spearman corr: 0.529, coverage: 0.760


In [16]:
draw_embeddings(window7, 30, html_file_name='word2vec_w7_p30')

In [16]:
word2vec_4 = Word2Vec.from_pretrained('data/word2vec/bigger_x2_window')
print(f'Syntactic coherence: {eval_syntactic_relations(word2vec_4):.3f}')
print(f'Semantic coherence: {eval_semantic_relations(word2vec_4):.3f}')

Syntactic coherence: 0.280
Semantic coherence: 0.043


In [19]:
points_4 = TSNE(perplexity=10).fit_transform(word2vec_4._embeddings)
draw_semantic_space(points_4[:, 0], points_4[:, 1], label=word2vec_4._vocab.words)

figure(id='p1256', ...)